In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from groundingdino.util.inference import load_model, load_image, predict, annotate

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.schemas import Box3D, Box2D
from src.common.nuscenes_utils import (
    convert_global_bbox_to_ego,
    get_sample_data_bboxes,
    filter_boxes_in_camera_fov,
    convert_3d_box_to_2d_box,
    CATEGORY_MAPPING_TO_UNIAD
)
from src.common.visualization import (
    plot_3d_boxes_on_image,
    plot_2d_boxes_on_image,
    TABLEAU10_NAMES
)
from src.grounding_dino.inference import predict_multi_labels

# Resolve paths relative to this notebook directory
CONFIG_PATH = ROOT / "GroundingDINO" / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
WEIGHTS_PATH = ROOT / "GroundingDINO" / "weights/groundingdino_swinb_cogcoor.pth"
IMAGE_PATH = ROOT / "GroundingDINO" / ".asset/cat_dog.jpeg"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create the model and load the weights
model = load_model(str(CONFIG_PATH), str(WEIGHTS_PATH), device=device)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

categories = {category["token"]: category for category in categories}
category_conversion = {k: v["category_name"] for k, v in CATEGORY_MAPPING_TO_UNIAD.items()}
category_names = list(dict.fromkeys(
    mapping["category_name"]
    for mapping in sorted(
        CATEGORY_MAPPING_TO_UNIAD.values(),
        key=lambda mapping: mapping["id"],
    )
))

print(f"original_category_names: {[category['name'] for category in categories.values()]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Select the scenes and camera channel and inference with GroundingDINO model
SCENE_NAME = "scene-0002"
CAMERA_CHANNEL = "CAM_FRONT"

# Define the box thresholds for each category group
BOX_THRESHOLDS = {
    'vehicle': [0.25, 0.35, 0.45],
    'road_object': [0.20, 0.30, 0.40],
    'two_wheeler': [0.20, 0.30, 0.40],
    'pedestrian': [0.20, 0.30, 0.40],
}

scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
# Select the samples
samples = [sample for sample in samples_all if sample["scene_token"] == scene["token"]]
print(f"num_samples: {len(samples)}")
sample_tokens = set(sample["token"] for sample in samples)
# Select the sample_data, ego_poses, and calibrated_sensors in the scene
sample_data = {sd["token"]: sd for sd in sample_data_all if sd["sample_token"] in sample_tokens}
sample_data = {sd_token: sd for sd_token, sd in sample_data.items() if sd["is_key_frame"]}  # Filter by is_key_frame
sample_data = dict(sorted(sample_data.items(), key=lambda item: item[1]["timestamp"]))  # sort by timestamp
ego_pose_tokens = set(sd["ego_pose_token"] for sd in sample_data.values())
ego_poses = {ep["token"]: ep for ep in ego_poses_all if ep["token"] in ego_pose_tokens}
calibrated_sensor_tokens = set(sd["calibrated_sensor_token"] for sd in sample_data.values())
calibrated_sensors = {cs["token"]: cs for cs in calibrated_sensors_all if cs["token"] in calibrated_sensor_tokens}
# Select the sample_annotations and instances in the scene
sample_annotations = {sa["token"]: sa for sa in sample_annotations_all if sa["sample_token"] in sample_tokens}
instance_tokens = set(sa["instance_token"] for sa in sample_annotations.values())
instances = {inst["token"]: inst for inst in instances_all if inst["token"] in instance_tokens}
# Create tracking_ids from instance tokens
tracking_ids = {inst_token: i for i, inst_token in enumerate(instance_tokens)}

# Filter sample_data to only include the selected camera channel
calibrated_sensors = {cs_token: cs for cs_token, cs in calibrated_sensors.items() if cs["sensor_token"] == sensor_lookup[CAMERA_CHANNEL]}
sample_data = {sd_token: sd for sd_token, sd in sample_data.items() if sd["calibrated_sensor_token"] in calibrated_sensors}
ego_poses = {ep_token: ep for ep_token, ep in ego_poses.items() if ep_token in set(sd["ego_pose_token"] for sd in sample_data.values())}

def get_gt_boxes(
    sd: dict[str, dict],
    ego_poses: dict[str, dict],
    calibrated_sensors: dict[str, dict],
    sample_annotations: dict[str, dict],
    instances: dict[str, dict],
    categories: dict[str, dict],
    category_mapping: dict[str, str],
    tracking_ids: dict[str, int]
) -> tuple[list[Box3D], list[Box2D]]:
    # Get the ground truth bounding boxes
    ego_pose = ego_poses[sd["ego_pose_token"]]
    image_width = sd["width"]
    image_height = sd["height"]
    camera_translation = calibrated_sensors[sd["calibrated_sensor_token"]]["translation"]
    camera_rotation = calibrated_sensors[sd["calibrated_sensor_token"]]["rotation"]
    camera_intrinsic = calibrated_sensors[sd["calibrated_sensor_token"]]["camera_intrinsic"]
    boxes_3d = get_sample_data_bboxes(sd, sample_annotations, instances, categories,
                                      category_conversion={k: v['category_name'] for k, v in category_mapping.items()},
                                      tracking_ids=tracking_ids)
    boxes_3d_ego = [convert_global_bbox_to_ego(box, ego_pose["translation"], ego_pose["rotation"]) for box in boxes_3d]
    valid_boxes_3d_ego = filter_boxes_in_camera_fov(boxes_3d_ego, camera_translation, camera_rotation,
                                                    camera_intrinsic, image_width, image_height)
    boxes_2d = [convert_3d_box_to_2d_box(box, camera_translation, camera_rotation, camera_intrinsic, image_width, image_height)
                for box in valid_boxes_3d_ego]
    boxes_2d_filtered = [box for box in boxes_2d if box is not None]
    
    return valid_boxes_3d_ego, boxes_2d_filtered

def filter_category_group_data(
    boxes_3d: list[Box3D],
    boxes_2d: list[Box2D],
    category_group: str,
    category_mapping: dict[str, dict],
) -> tuple[list[Box3D], list[Box2D]]:
    category_names_in_group = set([v['category_name'] for v in category_mapping.values() 
                                   if v['category_group'] == category_group])
    filtered_boxes_3d = [box for box in boxes_3d if box.label in category_names_in_group]
    filtered_boxes_2d = [box for box in boxes_2d if box.label in category_names_in_group]
    return category_names_in_group, filtered_boxes_3d, filtered_boxes_2d

# Iterate the first three sample data entries for the selected scene
for i, (token, sd) in enumerate(sample_data.items()):
    if i >= 3:
        break
    # Get the ground truth bounding boxes
    gt_boxes_3d, gt_boxes_2d = get_gt_boxes(
        sd=sd,
        ego_poses=ego_poses,
        calibrated_sensors=calibrated_sensors,
        sample_annotations=sample_annotations,
        instances=instances,
        categories=categories,
        category_mapping=CATEGORY_MAPPING_TO_UNIAD,
        tracking_ids=tracking_ids
    )
    # Load the image for inference
    image_path = NUSCENES_ROOT / sd["filename"]
    image = Image.open(image_path)
    # Create canvas for plotting the results
    category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])
    num_cols = 1 + len(list(BOX_THRESHOLDS.values())[0])  # +1 for the ground truth boxes
    num_rows = len(category_groups)  # Number of category groups
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6 * num_cols, 4 * num_rows))
    # Iterate the categories group
    for category_group_index, category_group in enumerate(category_groups):
        # Get the ground truth boxes for the category group
        category_names_in_group, category_gt_boxes_3d, category_gt_boxes_2d = filter_category_group_data(
            boxes_3d=gt_boxes_3d,
            boxes_2d=gt_boxes_2d,
            category_group=category_group,
            category_mapping=CATEGORY_MAPPING_TO_UNIAD
        )
        print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {len(category_gt_boxes_2d)}")
        # Plot the ground truth boxes for the category group
        category_colors = {category_name: TABLEAU10_NAMES[i % len(TABLEAU10_NAMES)]
                           for i, category_name in enumerate(category_names_in_group)}
        plot_2d_boxes_on_image(image, category_gt_boxes_2d,
                               ax=axes[category_group_index, 0],
                               color=category_colors,
                               title=f"category_group:{category_group}, GT boxes")
        # Iterate over different box thresholds
        for box_threshold_index, box_threshold in enumerate(BOX_THRESHOLDS[category_group]):
            # Infer the image with GroundingDINO model
            predicted_boxes, caption = predict_multi_labels(
                model=model,
                image=image,
                labels=category_names_in_group,
                box_threshold=box_threshold,
            )
            print(f"Detected {len(predicted_boxes)} boxes above the threshold of {box_threshold}")
            # convert box coordinates from normalized to pixel coordinates
            for box in predicted_boxes:
                box.xyxy = box.xyxy * np.array([image.width, image.height, image.width, image.height])
            plot_2d_boxes_on_image(image, predicted_boxes,
                                   ax=axes[category_group_index, box_threshold_index + 1],
                                   color=category_colors,
                                   title=f"{category_group}, box_threshold:{box_threshold}")
    plt.show()

In [ ]:
# SAM2 instance segmentation
# Iterate the first three sample data entries for the selected scene
for i, (token, sd) in enumerate(sample_data.items()):
    if i >= 3:
        break
    # Get the ground truth bounding boxes
    gt_boxes_3d, gt_boxes_2d = get_gt_boxes(
        sd=sd,
        ego_poses=ego_poses,
        calibrated_sensors=calibrated_sensors,
        sample_annotations=sample_annotations,
        instances=instances,
        categories=categories,
        category_mapping=CATEGORY_MAPPING_TO_UNIAD,
        tracking_ids=tracking_ids
    )
    # Load the image for inference
    image_path = NUSCENES_ROOT / sd["filename"]
    image = Image.open(image_path)
    # Create canvas for plotting the results
    category_groups = set([v['category_group'] for v in CATEGORY_MAPPING_TO_UNIAD.values()])
    num_cols = 2
    num_rows = len(category_groups)  # Number of category groups
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(8 * num_cols, 5 * num_rows))
    # Iterate the categories group
    for category_group_index, category_group in enumerate(category_groups):
        # Get the ground truth boxes for the category group
        category_names_in_group, category_gt_boxes_3d, category_gt_boxes_2d = filter_category_group_data(
            boxes_3d=gt_boxes_3d,
            boxes_2d=gt_boxes_2d,
            category_group=category_group,
            category_mapping=CATEGORY_MAPPING_TO_UNIAD
        )
        print(f"Category Group: {category_group}, Categories: {category_names_in_group}, Number of GT boxes: {len(category_gt_boxes_2d)}")
        # Plot the ground truth boxes for the category group
        category_colors = {category_name: TABLEAU10_NAMES[i % len(TABLEAU10_NAMES)]
                           for i, category_name in enumerate(category_names_in_group)}
        plot_2d_boxes_on_image(image, category_gt_boxes_2d,
                               ax=axes[category_group_index, 0],
                               color=category_colors,
                               title=f"category_group:{category_group}, GT boxes")
        ###### Grounding DINO Inference ######
        # Infer the image with GroundingDINO model
        predicted_boxes, caption = predict_multi_labels(
            model=model,
            image=image,
            labels=category_names_in_group,
            box_threshold=BOX_THRESHOLDS[category_group][1],  # Use the second threshold
        )
        print(f"Detected {len(predicted_boxes)} boxes above the threshold of {BOX_THRESHOLDS[category_group][1]}")
        # convert box coordinates from normalized to pixel coordinates
        for box in predicted_boxes:
            box.xyxy = box.xyxy * np.array([image.width, image.height, image.width, image.height])
        plot_2d_boxes_on_image(image, predicted_boxes,
                                ax=axes[category_group_index, 1],
                                color=category_colors,
                                title=f"{category_group}, box_threshold:{BOX_THRESHOLDS[category_group][1]}")
        ###### SAM2 Inference ######
    plt.show()